⚠️ **COMPUTE WARNING: CLOUD GPU REQUIRED** ⚠️

This notebook compiles a state-of-the-art Temporal Fusion Transformer (TFT) utilizing PyTorch Forecasting. The architecture leverages Variable Selection Networks and Interpretable Multi-Head Attention to process cyclical grid telemetry.

**Do not run this notebook on a local machine with integrated graphics.** 
Attempting to backpropagate this attention-based model across 46,000+ hourly records on a standard CPU will result in severe thermal throttling, system lockups, or Out-Of-Memory (OOM) errors. 

**Execution Instructions:**
1. Upload this notebook to Google Colab, Kaggle, or an AWS SageMaker instance.
2. Ensure a hardware accelerator (T4, A100, or V100 GPU) is attached to the runtime.
3. Mount the `tft_ready_data.csv` matrix into the environment before executing the data loaders.

In [ ]:
# Carry out the installations first.

!pip install pytorch-forecasting pytorch-lightning 

In [ ]:

import pandas as pd
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.metrics import QuantileLoss


# 1. Load the CSV
data_dataframe = pd.read_csv("tft_ready_data.csv")

# 2. PyTorch Forecasting Data Type Correction
# Categorical columns MUST be strings for the Variable Selection Networks
categorical_columns = ["month", "dayofweek", "hour", "region"]
for col in categorical_columns:
    data_dataframe[col] = data_dataframe[col].astype(str)

# 3. Define your training dataset boundaries
training_dataset = TimeSeriesDataSet(
    data_dataframe,  
    time_idx="time_idx",  
    target="normalized_target",
    group_ids=["region"],  
    max_encoder_length=168,  
    max_prediction_length=24,  
    static_categoricals=[],
    static_reals=[],
    time_varying_known_categoricals=["month", "dayofweek", "hour"],
    time_varying_known_reals=["Northern_Region_Avg_T2M"],  
    time_varying_unknown_reals=["normalized_target", "load_lag_1"],
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

print("TimeSeriesDataSet successfully created!")

In [ ]:
from pytorch_forecasting import TemporalFusionTransformer

tft = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=0.03,
    hidden_size=64,  # Size of the hidden layers in the LSTM/Attention cells
    lstm_layers=2,
    attention_head_size=4,
    dropout=0.1,
    loss=QuantileLoss([0.1, 0.5, 0.9]),  # Natively outputs all three target bounds simultaneously
    reduce_on_plateau_patience=4,
)

In [ ]:
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping

trainer = pl.Trainer(
    max_epochs=30,
    accelerator="gpu",  # Force execution on Colab's T4/A100 GPU hardware
    devices=1,
    callbacks=[EarlyStopping(monitor="val_loss", patience=3)]
)

In [ ]:
# 1. Generate the Validation Dataset configuration
validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset, 
    data_dataframe, 
    predict=True, 
    stop_randomization=True
)

# 2. Construct the PyTorch DataLoaders for GPU batching
batch_size = 64  
train_dataloader = training_dataset.to_dataloader(train=True, batch_size=batch_size, num_workers=2)
val_dataloader = validation_dataset.to_dataloader(train=False, batch_size=batch_size * 2, num_workers=2)

# 3. Trigger the Deep Learning Training Loop
print("Initiating TFT GPU Training Loop...")
trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader
)

In [ ]:
import matplotlib.pyplot as plt

# 1. Retrieve the optimal model weights saved by the Early Stopping callback
best_model_path = trainer.checkpoint_callback.best_model_path
print(f"Loading optimal weights from: {best_model_path}")
best_tft = TemporalFusionTransformer.load_from_checkpoint(best_model_path)

# 2. Run the validation data through the optimized network to map the attention layers
print("Calculating attention weights and feature importance...")
raw_predictions = best_tft.predict(val_dataloader, mode="raw", return_x=True)

# 3. Extract and plot the Variable Selection Network (VSN) interpretations
interpretation = best_tft.interpret_output(raw_predictions.output, reduction="sum")
figs = best_tft.plot_interpretation(interpretation)

# Display & Download the graphs
plt.savefig("tft_encoder_importance.png", dpi=300)
plt.show()